# QFF 2026 ? Quantum-Safe Secure Channel

This notebook demonstrates the ML-KEM-768 and ML-DSA-65 secure messaging prototype, its security tests, and local software benchmark results.

**Note:** Local Windows measurements are not Processor A/B hardware results.

## 1. Run security tests

In [ ]:
import os, subprocess, sys
from pathlib import Path

project = Path.cwd()
if not (project / 'src' / 'test_secure_channel.py').exists():
    raise FileNotFoundError('Open this notebook from the repository root.')

env = os.environ.copy()
env['PYTHONPATH'] = str(project / 'src')
liboqs_bin = str(Path(os.environ.get('TEMP', '')) / 'liboqs-install' / 'bin')
env['PATH'] = liboqs_bin + os.pathsep + env.get('PATH', '')
subprocess.run([sys.executable, 'src/test_secure_channel.py'], env=env, check=True)

## 2. Run the local TCP messaging demo

In [ ]:
subprocess.run([sys.executable, 'src/network_demo.py'], env=env, check=True)

## 3. View saved local benchmark results

In [ ]:
import pandas as pd
results = pd.read_csv('results/tables/local_benchmark.csv')
display(results)

## 4. Limitations

- The saved performance results are from the local Windows software environment.
- Processor A/B cryptographic timings have not been measured.
- The TCP demonstration is not a production-ready authenticated network protocol.
- See `report.md` for the threat discussion, topology comparison, and measurement methodology.


In [ ]:

import json
from pathlib import Path
import pandas as pd
from IPython.display import display

# 1. Show Processor A/B configuration
with open("processors/processor_A.json", encoding="utf-8") as f:
    processor_a = json.load(f)
with open("processors/processor_B.json", encoding="utf-8") as f:
    processor_b = json.load(f)

display(pd.DataFrame([
    {
        "Processor": "A",
        "Qubits": processor_a.get("num_qubits", processor_a.get("qubit_count")),
        "Coupling edges": len(processor_a["coupling_map"]),
        "Basis gates": ", ".join(processor_a["basis_gates"]),
    },
    {
        "Processor": "B",
        "Qubits": processor_b.get("num_qubits", processor_b.get("qubit_count")),
        "Coupling edges": len(processor_b["coupling_map"]),
        "Basis gates": ", ".join(processor_b["basis_gates"]),
    },
]))

# 2. Show connectivity-sensitive transpilation results
connectivity_path = Path("results/tables/connectivity_test.json")
if connectivity_path.exists():
    connectivity = json.loads(connectivity_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([
        {"Processor": label, **values}
        for label, values in connectivity.items()
    ]))
else:
    print("Run src/test_processor_transpilation.py to generate connectivity results.")

# 3. Show noisy simulation summary
noisy_path = Path("results/tables/noisy_simulation.json")
if noisy_path.exists():
    noisy = json.loads(noisy_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([
        {
            "Processor": label,
            "Shots": values["shots"],
            "Circuit depth": values["circuit_depth"],
            "CX gates": values["gate_counts"].get("cx", 0),
            "SWAP gates": values["swap_count"],
            "Measurement outcomes": len(values["counts"]),
        }
        for label, values in noisy.items()
    ]))
else:
    print("Run src/test_noisy_simulation.py to generate noisy simulation results.")

print(
    "Note: these are simulated quantum-circuit results. "
    "They are not physical hardware measurements or cryptographic handshake timings."
)
